In [ ]:
import os
import re
from ase import Atoms
from ase.units import Ry
from ase.calculators.siesta import Siesta
from ase.calculators.siesta.parameters import Species, PAOBasisBlock
from src.structure import Perovskite
from src.cleanfiles import cleanFiles

In [ ]:
def get_PAO_block(dir):
    """Function to extract the PAO.Basis block from a basis.out file generated by SIESTA.
    Parameters:
    - dir: Directory containing the basis.out file
    Returns:
    - block_basis: String containing the PAO.Basis block extracted from the basis.out file
    """
    # Open the generated .out file to extract the basis block
    with open(os.path.join(dir, "basis.out")) as f:
        out = f.read()
    # Extract the ChemicalSpeciesLabel and PAO.Basis block from the output file
    blocks = re.search(
        r"%block ChemicalSpeciesLabel.*?%endblock PAO.Basis",
        out, re.S).group(0)
    # Split the block into ChemicalSpeciesLabel and PAO.Basis block and return the PAO.Basis block
    block_species, block_basis = blocks.split("\n\n", 1)
    return block_basis

In [ ]:
def modify_basis(block_in, auto_radii=True):
    """Function to modify the basis block from a basis.fdf file generated by SIESTA.
    For now, the function adds a polarization orbital to the A-site (Ba) in a perovskite structure.
    Parameters:
    - block_in: String containing the basis block to be modified
    - auto_radii: Boolean indicating whether to set rcut and rmatch to 0.0 (default: True)
    Returns:
    - block_out: String containing the modified basis block
    """

    # Extract species labels (A.1, B.2, C.3)
    species = re.findall(r"\b[A-Z][a-zA-Z]*\.\d+\b", block_in)

    # Split basis block into blocks for each species
    positions = [block_in.index(s) for s in species]
    positions.append(len(block_in))
    blocks = [
        block_in[positions[i]:positions[i+1]]
        for i in range(len(species))
    ]
    # Remove the last line from the last block (contains %endblock PAO.Basis)
    blocks[2] = "\n".join(blocks[2].split("\n")[:-1])

    def _add_polarization_orb(block):
        """Helper function to add a polarization orbital to the A-site (Ba) in the basis block.
        Parameters:
        - block: String containing the basis block for the A-site (Ba)
        Returns:
        - block: String containing the modified basis block for the A-site (Ba) with the added polarization orbital
        """
        lines = block.rstrip().splitlines()

        # last orbital shell
        last_shell = lines[-3:]

        nums = list(map(int, re.findall(r"\d+", last_shell[0])))
        n, l, nz = nums[:3]

        pol_shell = [
            f" n={n-1}   {l+1}   1                         # n, l, Nzeta",
            last_shell[1],
            last_shell[2]
        ]

        # update number of l-shells
        header = lines[0].split()
        header[1] = f'{int(header[1]) + 1}  '
        lines[0] =  "                  ".join(header[0:3]) + " " + " ".join(header[3:])

        # append polarization shell
        lines.extend(pol_shell)

        block = "\n".join(lines) + "\n"
        return block
    
    blocks[0] = _add_polarization_orb(blocks[0])
    
    # Make new basis by joining the blocks together
    new_basis = "".join(blocks)
    # Split the input block into lines to extract the first 2 lines and the last line
    lines = block_in.splitlines()
    # Add the first 2 lines of the original basis block back to the top and the last line back to the end
    block_out = lines[0] + "\n" + lines[1] + "\n" + new_basis + "\n" + lines[-1]

    if auto_radii:
        block_out = re.sub(r"\b\d+\.\d+\b", "0.0", block_out)

    return block_out


In [ ]:
def generate_basis(perovskite, xcf='PBEsol', basis='DZP', EnergyShift=0.01, SplitNorm=0.15, dir=''):
    """Function to generate a basis.fdf file for a given perovskite structure using SIESTA.
    Parameters:
    - perovskite: Perovskite object containing the structure and properties of the material
    - xcf: Exchange-correlation functional to use (default: 'PBEsol')
    - basis: Basis set to use for the calculation (default: 'DZP').
             If basis ends with (lower-case) p, a polarization orbital will be added to the A-site (Ba)
    - EnergyShift: Energy shift for the basis functions in Ry (default: 0.01)
    - SplitNorm: Split norm for the basis functions (default: 0.15)
    - dir: Directory to save the input file (default: current working directory)
    """
    # Define current working directory and extract information from the perovskite object
    cwd = os.getcwd()
    #formula = perovskite.formula
    atoms = perovskite.atoms

    if basis.endswith('p'):
        basis = basis[:-1]
        add_p = True
    else:
        add_p = False

    # Calculation parameters in a dictionary
    calc_params = {
        'label': f'basis',
        'xc': xcf,
        'basis_set': basis,
        #'mesh_cutoff': MeshCutoff * Ry,
        'energy_shift': EnergyShift * Ry,
        #'kpts': kgrid,
        'directory': dir,
        'pseudo_path': os.path.join(cwd, 'pseudos', f'{xcf}')
    }
    # fdf arguments in a dictionary
    fdf_args = {
        #'PAO.BasisSize': basis,
        'PAO.SplitNorm': SplitNorm,
        'SCF.MustConverge' : 'F',
        'MaxSCFIterations': 0
        #'SCF.DM.Tolerance': 1e-6,
    }
    # Set up the Siesta calculator
    calc = Siesta(**calc_params, fdf_arguments=fdf_args)
    atoms.calc = calc
    # Perform a single-point calculation to generate the .out file containing the basis block
    atoms.get_potential_energy()

    # Remove intermediate files generated by the calculation, keeping only the important ones
    cleanFiles(directory=dir, confirm=False)

    # Extract the basis block from the generated .out file
    block = get_PAO_block(dir)

    # If the original basis string ended with 'p', add a polarization orbital to the A-site (Ba)
    if add_p:
        block = modify_basis(block)

    # Save block to a basis.fdf file in the specified directory
    with open(os.path.join(dir, "basis.fdf"), "w") as f:
        f.write(block)

    #calc.write_input(atoms, 'energies')
    

In [ ]:
perov = Perovskite('SrTiO3', bulk=True)

In [ ]:
generate_basis(perov, basis='DZPp', EnergyShift=0.001, SplitNorm=0.15, dir='resultsold/fdf')

In [ ]:
def run_test_calc(perovskite, xcf='PBEsol', basis='DZP', EnergyShift=0.01, SplitNorm=0.15,
              MeshCutoff=200, kgrid=(10, 10, 10), dir=''):


    # Define current working directory and extract information from the perovskite object
    cwd = os.getcwd()
    formula = perovskite.formula
    atoms = perovskite.atoms
    # Convert kgrid to a list to allow for modification
    kgrid = list(kgrid)

    # Calculation parameters in a dictionary
    calc_params = {
        'label': f'{formula}',
        'xc': xcf,
        'basis_set': basis,
        'mesh_cutoff': MeshCutoff * Ry,
        'energy_shift': EnergyShift * Ry,
        'kpts': kgrid,
        'directory': dir,
        'pseudo_path': os.path.join(cwd, 'pseudos', f'{xcf}')
    }
    dir_fdf = os.path.join(cwd, os.path.dirname(dir))
    # fdf arguments in a dictionary
    fdf_args = {
        '%include': os.path.join(dir_fdf, 'basis.fdf'),
        'PAO.SplitNorm': SplitNorm,
        #'SCF.DM.Tolerance': 1e-6
    }
    # Set up the Siesta calculator
    calc = Siesta(**calc_params, fdf_arguments=fdf_args)
    atoms.calc = calc
    # Perform a single-point calculation to generate the .fdf file and extract the basis block
    atoms.get_potential_energy()

    # Remove intermediate files generated by the calculation, keeping only the important ones
    cleanFiles(directory=dir, formats=['.DM', '.FA', '.XV'], confirm=False)

In [ ]:
dir = 'resultsold/fdf'
# Go up one folder from the directory
dir = os.path.dirname(dir)
dir

In [ ]:
run_test_calc(perov, xcf='PBEsol', MeshCutoff=200, kgrid=(4, 4, 4), dir='resultsold/fdf')